# COPT-MT: Paired Neural-Guidance Pilot (Milestone M1)

Notebook này thực hiện thực nghiệm M1: Đo lường lợi ích thực tế của neural guidance (checkpoint COPT-MIS) trong exact Maximum Clique search sau khi trừ chi phí trích xuất đặc trưng và inference.

> **Lưu ý quan trọng:** Hãy bật GPU trong **Notebook settings -> Accelerator -> GPU T4 (hoặc P100)** trước khi chạy!

In [ ]:
# 1. Kiểm tra môi trường GPU
import torch
import sys

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA version:", torch.version.cuda)
    print("Device name:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("Vui lòng bật GPU trong Notebook Settings (Accelerator -> GPU T4)")

In [ ]:
# 2. Clone mã nguồn nếu session chưa có repo; không xóa thư mục đang tồn tại
from pathlib import Path
import subprocess
repo = Path('/kaggle/working/COPT-MT')
if not repo.exists():
    subprocess.check_call(['git', 'clone', 'https://github.com/ghugn/COPT.git', str(repo)])
else:
    print('Dùng repo hiện có:', repo)
%cd /kaggle/working/COPT-MT
!git log -n 1 --oneline

In [ ]:
# 3. Cài đặt các thư viện phụ thuộc của COPT
%pip install -q \
    hydra-core \
    lightning \
    torchmetrics \
    torch-geometric \
    hydra-submitit-launcher rootutils networkx numba POT rich wandb \
    yacs einops dwave-networkx dimod ogb performer-pytorch python-sat

In [ ]:
# 4. Cài đặt PyG extensions (torch-scatter, torch-sparse) đúng bản Torch/CUDA
import subprocess
import sys
import torch

torch_version = torch.__version__.split("+")[0]
cuda_tag = "cu" + torch.version.cuda.replace(".", "")
wheel_index = f"https://data.pyg.org/whl/torch-{torch_version}+{cuda_tag}.html"
print("Đang tải wheel từ:", wheel_index)

cmd = [
    sys.executable, "-m", "pip", "install", "-q",
    "--no-index", f"--find-links={wheel_index}",
    "torch-scatter", "torch-sparse"
]
subprocess.check_call(cmd)
print("Đã cài đặt xong torch-scatter và torch-sparse thành công!")

In [ ]:
# 5. Giải nén bundle (code experiments/guidance và checkpoint epoch_189.ckpt)
from pathlib import Path
import zipfile
import shutil

%cd /kaggle/working/COPT-MT

# Tìm file bundle trong /kaggle/input hoặc /kaggle/working
bundle_matches = list(Path("/kaggle/input").rglob("*guidance*bundle*.zip")) + \
                 list(Path("/kaggle/working").glob("*guidance*bundle*.zip"))

if bundle_matches:
    bundle_path = bundle_matches[0]
    print(f"Tìm thấy bundle tại: {bundle_path}")
    with zipfile.ZipFile(bundle_path) as z:
        z.extractall("/kaggle/working/COPT-MT")
    print("Đã giải nén bundle thành công vào /kaggle/working/COPT-MT")
else:
    print("Không tìm thấy file zip bundle trong Input. Đang kiểm tra checkpoint rời...")

# Xác định đường dẫn checkpoint
ckpt_matches = list(Path("/kaggle/working/COPT-MT").rglob("*189*.ckpt")) + \
               list(Path("/kaggle/input").rglob("*189*.ckpt"))

assert len(ckpt_matches) > 0, "CHƯA TÌM THẤY CHECKPOINT epoch_189.ckpt trong bundle hoặc Kaggle Input."
CHECKPOINT = str(ckpt_matches[0])
print("Đường dẫn checkpoint đang dùng:", CHECKPOINT)

In [ ]:
# 6. Kiểm tra tính đúng đắn toán học của Solver (Unit Tests)
%cd /kaggle/working/COPT-MT
!python -m unittest experiments.guidance.test_solver -v

In [ ]:
# 7. Bước 1: Smoke test COPT trên GPU (xác thực model load & inference)
%cd /kaggle/working/COPT-MT
!rm -rf /kaggle/working/guidance_smoke_copt
!python -u -m experiments.guidance.pilot \
    --checkpoint "{CHECKPOINT}" \
    --device cuda \
    --sizes 30 40 \
    --densities 0.5 0.7 \
    --graphs-per-cell 1 \
    --states-per-graph 2 \
    --repeats 2 \
    --seconds 1 \
    --output /kaggle/working/guidance_smoke_copt

from pathlib import Path
smoke_summary = Path("/kaggle/working/guidance_smoke_copt/summary.json")
if smoke_summary.exists():
    print("Smoke test hoàn thành! Summary:")
    print(smoke_summary.read_text())

In [ ]:
# 8. Bước 2: Chạy thí nghiệm chính Milestone M1 (Paired Replay Benchmark)
# Thiết lập: 60 đồ thị ER (sizes 40, 60, 80, 120; densities 0.3, 0.5, 0.7)
# So sánh cùng state, cùng incumbent giữa Structural vs Random vs COPT
%cd /kaggle/working/COPT-MT
!rm -rf /kaggle/working/guidance_pilot_m1
!python -u -m experiments.guidance.pilot \
    --checkpoint "{CHECKPOINT}" \
    --device cuda \
    --sizes 40 60 80 120 \
    --densities 0.3 0.5 0.7 \
    --graphs-per-cell 5 \
    --states-per-graph 4 \
    --repeats 3 \
    --seconds 2 \
    --node-limit 200000 \
    --output /kaggle/working/guidance_pilot_m1

In [ ]:
# 9. Tải và đóng gói kết quả M1
import shutil
from pathlib import Path
from IPython.display import FileLink, display

output = Path("/kaggle/working/guidance_pilot_m1")
if output.exists():
    summary_file = output / "summary.json"
    if summary_file.exists():
        print("=== KẾT QUẢ SUMMARY M1 ===")
        print(summary_file.read_text())
    archive = shutil.make_archive("/kaggle/working/guidance_pilot_m1", "zip", output)
    print(f"Đã tạo file zip kết quả: {archive}")
    display(FileLink(archive))
    print("Bạn có thể click vào link trên hoặc tải file guidance_pilot_m1.zip từ panel Kaggle Output!")
else:
    print("Không tìm thấy thư mục output, kiểm tra lại log bước 8.")